#CIS335 Assignment 3

Continuation of the Assignment 2 Loan Approval Analysis

Tasks:
- Use the Bagging Classifier to analyze loan approval
- Use Random Forest Classifier to analyze loan approval
- Compare which method has the highest accuracy, precision, recall, and F2 score


#Imports

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from numpy import ravel
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import Normalizer, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score
from sklearn.datasets import make_classification
from sklearn.metrics import fbeta_score

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Loan-Approval-Dataset.csv to Loan-Approval-Dataset.csv


#Initial DataFrame Initialization

takes the Loan Approval Prediction Dataset and converts it to a pandas DataFrame

Attributes:
- Gender
- Married
- Dependents
- Education
- Self_Employed
- ApplicantIncome
- CoapplicantIncome
- LoanAmount
- Loan_Amount_Term
- Credit_History
- Property_Area
- Loan_Status

In [7]:
df = pd.read_csv("Loan-Approval-Dataset.csv")
df = df.drop(columns=["Loan_ID"])
df

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...
609,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


#Features and Target

**Target:** Loan Status

**Features:** Gender, Married, Dependents, Education, Self_employed, ApplicantIncome, CoapplicantIncome, LoanAmount, Loan_Amount_Term, Credit_History, Property_Area

In [9]:
from numpy._core.arrayprint import format_float_scientific
# Features
X = df.iloc[:, 1:-1]

# Converts all Categorical columns to unique integers
X = pd.get_dummies(X, drop_first=False)

# Target: Loan Status
y = df.iloc[:, -1]

# Encode y to unique integers: 1 = Y, 0 = N
encoder = LabelEncoder()
y = encoder.fit_transform(ravel(y))

#Split Data

In [12]:
# Split Data: Train 80% Test 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=90)

# Split Train Data: Train 90% Validation 10%
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, shuffle=True, random_state=90)

#Part 1 - Bagging Classifier



#Normalization Pipeline

Three Normalizers:
- MinMax
- Z-Score
- None

In [231]:
n_values = [5, 10, 20, 50, 100]

gb_pipes = {}

for scaler_name, scaler in {
    "none": None,
    "zscore": StandardScaler(),
    "minmax": MinMaxScaler()
}.items():

    for n in n_values:
        steps = []

        if scaler is not None:
            steps.append(('scaler', scaler))

        steps.append(('selector', VarianceThreshold()))
        steps.append(('classifier', BaggingClassifier(n_estimators=n, random_state=0)))

        pipe_name = f"{scaler_name}_{n}"
        gb_pipes[pipe_name] = Pipeline(steps)


#Run Bagging Classifier Pipeline

In [237]:
gb_results = []


for name, pipe in gb_pipes.items():
  pipe.fit(X_train, y_train)
  y_pred = pipe.predict(X_test)

  scaler, n = name.split('_')

  gb_results.append({
      'Name': name,
      'Scaler': scaler,
      'N': int(n),
      'Accuracy': pipe.score(X_test, y_test),
      'Precision': precision_score(y_test, y_pred),
      'Recall': recall_score(y_test, y_pred),
      'F2': fbeta_score(y_test, y_pred, beta=2)
  })


gb_df = pd.DataFrame(gb_results)
print(gb_df)

          Name  Scaler    N  Accuracy  Precision    Recall        F2
0       none_5    none    5  0.756098   0.765306  0.914634  0.880282
1      none_10    none   10  0.756098   0.765306  0.914634  0.880282
2      none_20    none   20  0.747967   0.762887  0.902439  0.870588
3      none_50    none   50  0.747967   0.757576  0.914634  0.878220
4     none_100    none  100  0.756098   0.760000  0.926829  0.887850
5     zscore_5  zscore    5  0.756098   0.765306  0.914634  0.880282
6    zscore_10  zscore   10  0.756098   0.765306  0.914634  0.880282
7    zscore_20  zscore   20  0.747967   0.762887  0.902439  0.870588
8    zscore_50  zscore   50  0.747967   0.757576  0.914634  0.878220
9   zscore_100  zscore  100  0.756098   0.760000  0.926829  0.887850
10    minmax_5  minmax    5  0.764228   0.773196  0.914634  0.882353
11   minmax_10  minmax   10  0.756098   0.765306  0.914634  0.880282
12   minmax_20  minmax   20  0.747967   0.762887  0.902439  0.870588
13   minmax_50  minmax   50  0.747

#Part 2 - Random Forest Classifier

#Normalization Pipeline

Three Normalizers:
- MinMax
- Z-Score
- None

In [223]:
n2_values = [5, 10]
depth_values = [3, 5]

rf_pipes = {}

for scaler_name, scaler in {
    "none": None,
    "zscore": StandardScaler(),
    "minmax": MinMaxScaler()
}.items():

    for n in n2_values:
      for d in depth_values:
        steps = []

        if scaler is not None:
            steps.append(('scaler', scaler))

        steps.append(('selector', VarianceThreshold()))
        steps.append(('classifier', RandomForestClassifier(n_estimators=n, max_depth=d)))

        pipe_name = f"{scaler_name}_{n}_{d}"
        rf_pipes[pipe_name] = Pipeline(steps)

#Random Forest Classifier Pipeline

In [236]:
rf_results = []

for name, pipe in rf_pipes.items():

    # Train
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    # Metrics
    rf_score = pipe.score(X_test, y_test)
    rf_prec = precision_score(y_test, y_pred)
    rf_recall = recall_score(y_test, y_pred)
    rf_f2 = fbeta_score(y_test, y_pred, beta=2)
    rf_cm = confusion_matrix(y_test, y_pred)

    scaler, n, depth = name.split('_')

    rf_results.append({
        'Name': name,
        'Scaler': scaler,
        'N': int(n),
        'Depth': int(depth),
        'Accuracy': pipe.score(X_test, y_test),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F2': fbeta_score(y_test, y_pred, beta=2)
    })


rf_df = pd.DataFrame(rf_results)
print(rf_df)



           Name  Scaler   N  Depth  Accuracy  Precision    Recall        F2
0      none_5_3    none   5      3  0.739837   0.723214  0.987805  0.920455
1      none_5_5    none   5      5  0.747967   0.738318  0.963415  0.908046
2     none_10_3    none  10      3  0.780488   0.757009  0.987805  0.931034
3     none_10_5    none  10      5  0.772358   0.750000  0.987805  0.928899
4    zscore_5_3  zscore   5      3  0.772358   0.750000  0.987805  0.928899
5    zscore_5_5  zscore   5      5  0.739837   0.723214  0.987805  0.920455
6   zscore_10_3  zscore  10      3  0.707317   0.698276  0.987805  0.912162
7   zscore_10_5  zscore  10      5  0.756098   0.740741  0.975610  0.917431
8    minmax_5_3  minmax   5      3  0.780488   0.757009  0.987805  0.931034
9    minmax_5_5  minmax   5      5  0.764228   0.752381  0.963415  0.912240
10  minmax_10_3  minmax  10      3  0.780488   0.757009  0.987805  0.931034
11  minmax_10_5  minmax  10      5  0.764228   0.747664  0.975610  0.919540


#Part 3 - Comparison

In [253]:
# Compare Averages Between All Pipes
bg_acc_avg = gb_df['Accuracy'].mean()
bg_prec_avg = gb_df['Precision'].mean()
bg_recall_avg = gb_df['Recall'].mean()
bg_f2_avg = gb_df['F2'].mean()

rf_acc_avg = rf_df['Accuracy'].mean()
rf_prec_avg = rf_df['Precision'].mean()
rf_recall_avg = rf_df['Recall'].mean()
rf_f2_avg = rf_df['F2'].mean()

print(f'Gradient Boosting Averages:\n\nAccuracy: {bg_acc_avg}\nPrecision: {bg_prec_avg}\nRecall: {bg_recall_avg}\nF2 Score: {bg_f2_avg}\n\n')
print(f'Random Forest Averages:\n\nAccuracy: {rf_acc_avg}\nPrecision: {rf_acc_avg}\nRecall: {rf_recall_avg}\nF2 Score: {rf_f2_avg}\n\n')

avg_results = {
    'Accuracy': "Gradient Boosting" if bg_acc_avg > rf_acc_avg else "Random Forest",
    'Precision': "Gradient Boosting" if bg_prec_avg > rf_prec_avg else "Random Forest",
    'Recall': "Gradient Boosting" if bg_recall_avg > rf_recall_avg else "Random Forest",
    'F2': "Gradient Boosting" if bg_f2_avg > rf_f2_avg else "Random Forest"
}




print(f'Highest Averages:\n\nAccuracy: {avg_results['Accuracy']}\nPrecision: {avg_results['Precision']}\nRecall: {avg_results['Recall']}\nF2 Score: {avg_results['F2']}\n\n')


bg_count = sum(1 for v in avg_results.values() if v == "Gradient Boosting")
rf_count = sum(1 for v in avg_results.values() if v == "Random Forest")


print('Outcome:\n\n******', end=' ')

if bg_count > rf_count:
  print('Gradient Boosting has the best average performance', end=' ')
elif bg_count == rf_count:
  print('Both perform similarly', end=' ')
else:
  print('Random Forest has the best average performance', end=' ')

print('******\n\n')



# Compare Normalization Methods: None, MinMax, ZScore

scalers = ['minmax', 'zscore', 'none']
results_dict = {}

for scaler in scalers:
  results_dict[scaler] = {
  'Gradient Boosting': gb_df[gb_df['Scaler'] == scaler][['Accuracy','Precision','Recall','F2']].mean().to_dict(),
  'Random Forest': rf_df[rf_df['Scaler'] == scaler][['Accuracy','Precision','Recall','F2']].mean().to_dict()
  }

scaler_summary = {}

for scaler, models in results_dict.items():
  scaler_summary[scaler] = {
  metric: sum(m[metric] for m in models.values()) / len(models)
  for metric in ['Accuracy','Precision','Recall','F2']
  }

scaler_df = pd.DataFrame(scaler_summary)
print(f'Scaler Summary:\n\n{scaler_df}\n\n')

best_scaler = max(scaler_summary, key=lambda x: scaler_summary[x]['F2'])
print(f"****** Best Scaler: {best_scaler} ******")



# Compute Best Configuration

best_score = float('-inf')
best_config = None

for scaler, models in results_dict.items():
    for model_name, metrics in models.items():
        f2 = metrics.get('F2')

        if f2 is None or pd.isna(f2):
            continue

        if f2 > best_score:
            best_score = f2
            best_config = (scaler, model_name)

print(f"\n\nBest configuration:\n")
print(f"Scaler: {best_config[0]}")
print(f"Model: {best_config[1]}")
print(f"F2 Score: {best_score}")



Gradient Boosting Averages:

Accuracy: 0.7533875338753386
Precision: 0.7627409036716842
Recall: 0.9146341463414633
F2 Score: 0.8795825280785252


Random Forest Averages:

Accuracy: 0.7588075880758808
Precision: 0.7588075880758808
Recall: 0.9817073170731708
F2 Score: 0.9217692042332782


Highest Averages:

Accuracy: Random Forest
Precision: Gradient Boosting
Recall: Random Forest
F2 Score: Random Forest


Outcome:

****** Random Forest has the best average performance ******


Scaler Summary:

             minmax    zscore      none
Accuracy   0.763415  0.748374  0.756504
Precision  0.758654  0.745136  0.752175
Recall     0.946646  0.949695  0.948171
F2         0.901661  0.899591  0.900776


****** Best Scaler: minmax ******


Best configuration:

Scaler: minmax
Model: Random Forest
F2 Score: 0.9234623450399511
